# 00 — Quantize Llama-3-8B and save the demo checkpoints

Produces the checkpoints used by the recorded defense demo.

**Before running (Kaggle):**
1. Accelerator: **GPU T4 x2** - Internet: **On**.
2. Have a Hugging Face token ready (needs access to the gated
   `meta-llama/Meta-Llama-3-8B`). You will be prompted to paste it at run time;
   for non-interactive **Save & Run All** runs, add it instead as a Kaggle Secret
   named **`HF_TOKEN`** (the prompt is skipped automatically).
3. Set `VARIANT` in the next cell, then **Save Version -> Save & Run All**.

Kaggle caps notebook output at ~19.5 GB and each fp16 checkpoint is ~15 GB, so run this
notebook **twice** (one Save Version per variant) and turn each run's *Output* into its own
**private** Dataset:

| `VARIANT` | Output | Suggested dataset name |
|---|---|---|
| `"rtn"` | plain-RTN checkpoint (`rtn_xl.py`) + Demo-1 standalone artifacts | `llama3-rtn-demo` |
| `"rtn_reflip"` | RTN -> Flip -> GQA-ReFlip checkpoint (`rtn_gqa_xl.py --apply-gqa-reflip`) | `llama3-rtn-reflip-demo` |


In [ ]:
VARIANT = "rtn"   # <-- set to "rtn" or "rtn_reflip", then Save & Run All

assert VARIANT in ("rtn", "rtn_reflip"), "VARIANT must be 'rtn' or 'rtn_reflip'"
print("Building variant:", VARIANT)

In [ ]:
# --- Clone the repository and install dependencies (Kaggle already ships torch+CUDA) ---
import subprocess, sys, os

if not os.path.exists("/kaggle/tmp/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Itvc0110/Reflip-Flip-on-QKV-.git", "/kaggle/tmp/repo"], check=True)
os.chdir("/kaggle/tmp/repo")
print("repo at", os.getcwd())

# Keep Kaggle's preinstalled torch; install only what the pipeline needs on top.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.45,<4.53", "datasets>=2.20,<3.0", "accelerate",
                "sentencepiece", "kneed", "matplotlib", "seaborn", "tqdm", "psutil"], check=True)
print("deps ready")

In [ ]:
# --- Log in to Hugging Face and download the gated base model to scratch (NOT into output) ---
from getpass import getpass
from huggingface_hub import login, snapshot_download

# Primary: type/paste the token at the prompt (input hidden).
# Fallback (non-interactive "Save & Run All" runs): Kaggle Secret named HF_TOKEN.
try:
    token = getpass("Paste your Hugging Face access token (input hidden): ").strip()
except Exception:
    token = ""
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print("(using HF_TOKEN Kaggle Secret)")
login(token=token)

MODEL_DIR = "/kaggle/tmp/models/Llama-3-8B"
snapshot_download("meta-llama/Meta-Llama-3-8B", local_dir=MODEL_DIR,
                  ignore_patterns=["original/*", "*.pth"])
print("base model at", MODEL_DIR)

In [ ]:
# --- Quantize (this is the long cell) ---
import subprocess, sys

if VARIANT == "rtn":
    # Plain RTN baseline: no calibration pass needed (~30-60 min incl. model I/O)
    subprocess.run([sys.executable, "rtn_xl.py",
                    "--model-path", MODEL_DIR,
                    "--output-dir", "/kaggle/working/llama3_rtn",
                    "--group-size", "128"], check=True)
else:
    # RTN base (identity AWQ scale) -> Flip -> GQA ReFlip (~2.5-4 h: 128-sample calibration
    # forward passes dominate; the AWQ grid search itself is skipped by design)
    subprocess.run([sys.executable, "rtn_gqa_xl.py",
                    "--model-path", MODEL_DIR,
                    "--output-dir", "/kaggle/working/llama3_rtn_reflip",
                    "--n-calib", "128",
                    "--layer-batch-size", "16",
                    "--apply-gqa-reflip",
                    "--gqa-critical-dim-pct", "0.15",
                    "--gqa-max-flip-pct", "0.05"], check=True)
print("quantization done")

In [ ]:
# --- Demo-1 standalone case-study artifacts (only in the 'rtn' run; small files) ---
import subprocess, sys, os

if VARIANT == "rtn":
    subprocess.run([sys.executable, "xspot.py",
                    "--model-path", MODEL_DIR,
                    "--layer-id", "8", "--group-id", "3",
                    "--output-dir", "/kaggle/tmp/xspot_layer8_group3"], check=True)
    subprocess.run([sys.executable, "fast_quantize_qkv.py",
                    "--data-dir", "/kaggle/tmp/xspot_layer8_group3",
                    "--group-id", "3",
                    "--critical-dim-pct", "0.1",
                    "--output-dir", "/kaggle/working/demo1"], check=True)
    subprocess.run([sys.executable, "tools/summarize_qkv_results.py",
                    "--npz", "/kaggle/working/demo1/quantization_results.npz"], check=True)
    subprocess.run([sys.executable, "tools/plot_kneedle_sensitivity.py",
                    "--npz", "/kaggle/working/demo1/quantization_results.npz",
                    "--out", "/kaggle/working/demo1/reflip_sensitivity_kneedle.png"], check=True)
    subprocess.run([sys.executable, "tools/plot_flip_activation_kneedle.py",
                    "--npz", "/kaggle/working/demo1/quantization_results.npz",
                    "--out", "/kaggle/working/demo1/flip_activation_kneedle.png"], check=True)
    print("Demo-1 artifacts saved to /kaggle/working/demo1")
else:
    print("skipped (Demo-1 artifacts are produced in the 'rtn' run)")

In [ ]:
# --- Verify output and report sizes ---
import os
from transformers import AutoConfig

total = 0
for root, _, files in os.walk("/kaggle/working"):
    for f in files:
        total += os.path.getsize(os.path.join(root, f))
print(f"total output size: {total / 1e9:.2f} GB (Kaggle cap ~19.5 GB)")

ckpt = "/kaggle/working/llama3_rtn" if VARIANT == "rtn" else "/kaggle/working/llama3_rtn_reflip"
print(AutoConfig.from_pretrained(ckpt))
print("\nDone. From this notebook's Output tab: 'New Dataset' (keep it PRIVATE).")